In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("AbileneTrafficAnalysis")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Spark version: 4.2.0


In [12]:
path = r"C:\Users\samya\PyCharmProject\Predictive-SDN-Dynamic-Load-Balancer_data\data\ml\traffic_network_native.parquet"    
df = spark.read.parquet(path)
print (df.show(5))
print(df.printSchema())
print("Total number of rows:", df.count())

+---+--------------------+-------------------+-----------+----------------+---------+--------------+--------+------------+----------+-----------+---------------+-------------------+----------+--------------------+-----------------+---------------+------------+--------------------+
| id|            event_id|    event_timestamp|source_node|destination_node|source_ip|destination_ip|protocol|packet_count|byte_count|duration_ms|throughput_mbps|packet_loss_percent|latency_ms|         raw_payload|      kafka_topic|kafka_partition|kafka_offset|         ingested_at|
+---+--------------------+-------------------+-----------+----------------+---------+--------------+--------+------------+----------+-----------+---------------+-------------------+----------+--------------------+-----------------+---------------+------------+--------------------+
|  1|abilene_20040301-...|2004-03-01 05:30:00|     ATLAM5|          ATLAng|     NULL|          NULL|    NULL|        NULL|      NULL|       NULL|       0.

In [10]:
df.select(
    F.min('event_timestamp').alias("min_event_time"),
    F.max('event_timestamp').alias("max_event_time"),
    F.countDistinct('source_node').alias("unique_src_nodes"),
    F.countDistinct('destination_node').alias("unique_dst_nodes"),
    F.countDistinct("event_timestamp").alias("unique_timestamps"),
    F.count("*").alias("total_events")
).show()


+-------------------+-------------------+----------------+----------------+-----------------+------------+
|     min_event_time|     max_event_time|unique_src_nodes|unique_dst_nodes|unique_timestamps|total_events|
+-------------------+-------------------+----------------+----------------+-----------------+------------+
|2004-03-01 05:30:00|2004-09-11 05:25:00|              12|              12|            48096|     6015948|
+-------------------+-------------------+----------------+----------------+-----------------+------------+



In [15]:
od_df = (
    df
    .select(
        F.col("source_node").alias("src_node"),
        F.col("destination_node").alias("dst_node")
    )
    .distinct()
)

print("Unique source-destination pairs:", od_df.count())

od_df.orderBy(
    "src_node",
    "dst_node"
).show(200, truncate=False)

Unique source-destination pairs: 132
+--------+--------+
|src_node|dst_node|
+--------+--------+
|ATLAM5  |ATLAng  |
|ATLAM5  |CHINng  |
|ATLAM5  |DNVRng  |
|ATLAM5  |HSTNng  |
|ATLAM5  |IPLSng  |
|ATLAM5  |KSCYng  |
|ATLAM5  |LOSAng  |
|ATLAM5  |NYCMng  |
|ATLAM5  |SNVAng  |
|ATLAM5  |STTLng  |
|ATLAM5  |WASHng  |
|ATLAng  |ATLAM5  |
|ATLAng  |CHINng  |
|ATLAng  |DNVRng  |
|ATLAng  |HSTNng  |
|ATLAng  |IPLSng  |
|ATLAng  |KSCYng  |
|ATLAng  |LOSAng  |
|ATLAng  |NYCMng  |
|ATLAng  |SNVAng  |
|ATLAng  |STTLng  |
|ATLAng  |WASHng  |
|CHINng  |ATLAM5  |
|CHINng  |ATLAng  |
|CHINng  |DNVRng  |
|CHINng  |HSTNng  |
|CHINng  |IPLSng  |
|CHINng  |KSCYng  |
|CHINng  |LOSAng  |
|CHINng  |NYCMng  |
|CHINng  |SNVAng  |
|CHINng  |STTLng  |
|CHINng  |WASHng  |
|DNVRng  |ATLAM5  |
|DNVRng  |ATLAng  |
|DNVRng  |CHINng  |
|DNVRng  |HSTNng  |
|DNVRng  |IPLSng  |
|DNVRng  |KSCYng  |
|DNVRng  |LOSAng  |
|DNVRng  |NYCMng  |
|DNVRng  |SNVAng  |
|DNVRng  |STTLng  |
|DNVRng  |WASHng  |
|HSTNng  |ATLAM5  |
|HS

In [16]:
od_df = (
    df
    .select(
        F.col("source_node").alias("src_node"),
        F.col("destination_node").alias("dst_node"),
        F.col("event_timestamp").alias("event_time"),
        F.col("throughput_mbps").alias("traffic_volume")
    )
)

od_df.show(20, truncate=False)

+--------+--------+-------------------+--------------+
|src_node|dst_node|event_time         |traffic_volume|
+--------+--------+-------------------+--------------+
|ATLAM5  |ATLAng  |2004-03-01 05:30:00|0.522208      |
|ATLAM5  |SNVAng  |2004-03-01 05:30:00|0.747405      |
|ATLAng  |ATLAM5  |2004-03-01 05:30:00|0.445149      |
|ATLAng  |CHINng  |2004-03-01 05:30:00|16.283117     |
|ATLAng  |HSTNng  |2004-03-01 05:30:00|3.931403      |
|ATLAng  |KSCYng  |2004-03-01 05:30:00|4.844035      |
|ATLAng  |NYCMng  |2004-03-01 05:30:00|12.803955     |
|CHINng  |ATLAM5  |2004-03-01 05:30:00|3.793693      |
|CHINng  |ATLAng  |2004-03-01 05:30:00|12.735325     |
|DNVRng  |ATLAM5  |2004-03-01 05:30:00|0.230805      |
|DNVRng  |HSTNng  |2004-03-01 05:30:00|8.408763      |
|DNVRng  |LOSAng  |2004-03-01 05:30:00|21.375568     |
|DNVRng  |SNVAng  |2004-03-01 05:30:00|14.385464     |
|HSTNng  |ATLAng  |2004-03-01 05:30:00|2.956779      |
|HSTNng  |DNVRng  |2004-03-01 05:30:00|3.693019      |
|HSTNng  |

In [20]:
self_loops = df.filter(F.col("source_node") == F.col("destination_node"))
print("Number of self-loop events:", self_loops.count())

Number of self-loop events: 0


In [21]:
self_loops.select(
    "source_node",
    "destination_node"
).distinct().show()

+-----------+----------------+
|source_node|destination_node|
+-----------+----------------+
+-----------+----------------+



In [23]:
duplicate_events = (
    df
    .groupBy(
        "source_node",
        "destination_node",
        "event_timestamp"
    )
    .agg(
        F.count("*").alias("duplicate_count")
    )
    .filter(F.col("duplicate_count") > 1)
)
print("Number of duplicate events:", duplicate_events.count())
duplicate_events.orderBy(
    F.desc('duplicate_count')
).show(20, truncate=False)

Number of duplicate events: 0
+-----------+----------------+---------------+---------------+
|source_node|destination_node|event_timestamp|duplicate_count|
+-----------+----------------+---------------+---------------+
+-----------+----------------+---------------+---------------+



In [24]:
df.select(
    F.count("*").alias("rows"),
    F.count("throughput_mbps").alias("non_null_throughput"),
    F.sum(F.when(F.col("throughput_mbps").isNull(), 1).otherwise(0)).alias("null_throughput"),
    F.sum(F.when(F.col("throughput_mbps") < 0, 1).otherwise(0)).alias("negative_throughput"),
    F.sum(F.when(F.col("throughput_mbps") > 10000, 1).otherwise(0)).alias("out_of_range_throughput"),
    F.sum(F.when(F.col("throughput_mbps") == 0, 1).otherwise(0)).alias("zero_throughput"),
).show()

+-------+-------------------+---------------+-------------------+-----------------------+---------------+
|   rows|non_null_throughput|null_throughput|negative_throughput|out_of_range_throughput|zero_throughput|
+-------+-------------------+---------------+-------------------+-----------------------+---------------+
|6015948|            6015948|              0|                  0|                      0|              0|
+-------+-------------------+---------------+-------------------+-----------------------+---------------+



In [25]:
quantiles = df.approxQuantile(
    "throughput_mbps",
    [0.00, 0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1.00],
    0.001
)

for q, value in zip(
    [0.00, 0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1.00],
    quantiles
):
    print(f"{q:>6.3f}: {value:.6f}")

 0.000: 0.013333
 0.010: 0.026667
 0.050: 0.073243
 0.250: 1.899131
 0.500: 9.571627
 0.750: 26.224683
 0.900: 55.120187
 0.950: 84.917904
 0.990: 157.525275
 0.999: 7085.250667
 1.000: 7085.250667
